In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

In [ ]:
df=pd.read_csv('/content/Feature_engineered_price_tracking_data.csv')

In [ ]:
df = df.dropna()

In [ ]:
# Encode categorical variables
label_encoders = {}
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [ ]:
df.describe()

,name,main_category,sub_category,ratings,no_of_ratings,discount_price,actual_price,discounted_price_1,discounted_price_2,discounted_price_3,...,discounted_price_5,discounted_price_6,discounted_price_7,discount_percentage,price_ratio,popularity_score,price_difference,log_no_of_ratings,main_category_encoded,sub_category_encoded
count,533769.000000,533769.000000,533769.000000,533769.000000,533769.000000,5.337690e+05,5.337690e+05,5.337690e+05,5.337690e+05,5.337690e+05,...,5.337690e+05,5.337690e+05,5.337690e+05,533769.000000,5.337690e+05,533769.000000,5.337690e+05,533769.000000,533769.000000,533769.000000
mean,192526.624219,9.526018,56.924147,3.836930,836.843243,2.872740e+03,2.311141e+04,2.310935e+03,2.291829e+03,2.292148e+03,...,2.271428e+03,2.313082e+03,2.274189e+03,45.256731,5.474327e-01,17.476743,2.023867e+04,4.514099,9.526018,56.924147
std,111066.296860,6.854708,30.407881,0.835263,7097.350757,9.565800e+03,1.355086e+07,8.539157e+03,8.542802e+03,8.556883e+03,...,8.542727e+03,8.552172e+03,8.565542e+03,24.531598,2.453160e-01,10.255506,1.355086e+07,2.373919,6.854708,30.407881
min,0.000000,0.000000,0.000000,1.000000,1.000000,4.000000e+00,4.000000e+00,7.540000e+00,7.520000e+00,7.240000e+00,...,7.280000e+00,7.830000e+00,7.640000e+00,0.000000,5.949495e-08,0.693147,0.000000e+00,0.693147,0.000000,0.000000
25%,95603.000000,1.000000,35.000000,3.700000,9.000000,3.990000e+02,9.900000e+02,3.767500e+02,3.291600e+02,3.297000e+02,...,3.015000e+02,3.769000e+02,3.018600e+02,28.578655,3.541927e-01,8.464333,3.000000e+02,2.302585,1.000000,35.000000
50%,193559.000000,10.000000,58.000000,3.837409,117.000000,6.990000e+02,1.599000e+03,5.766700e+02,5.671700e+02,5.673300e+02,...,5.671700e+02,5.955300e+02,5.674100e+02,50.012503,4.998750e-01,18.518907,7.000000e+02,4.770685,10.000000,58.000000
75%,287181.000000,17.000000,85.000000,4.100000,840.778698,1.549000e+03,2.999000e+03,1.216030e+03,1.216800e+03,1.216590e+03,...,1.217230e+03,1.216410e+03,1.216870e+03,64.580726,7.142135e-01,25.846934,1.548000e+03,6.735517,17.000000,85.000000
max,384823.000000,19.000000,111.000000,100.000000,589547.000000,1.249990e+06,9.900000e+09,1.172584e+06,1.177847e+06,1.224681e+06,...,1.151593e+06,1.216494e+06,1.204944e+06,99.999994,1.000000e+00,673.551715,9.899999e+09,13.287111,19.000000,111.000000


In [ ]:
X = df.drop(columns=['actual_price'])
y = df['actual_price']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
xgb = XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
sgd = SGDRegressor(max_iter=1000, tol=1e-3)

In [ ]:
# Combine models using Voting Regressor
voting_regressor = VotingRegressor([('rf', rf), ('xgb', xgb), ('sgd', sgd)])

In [ ]:
# Train the ensemble model
voting_regressor.fit(X_train, y_train)

VotingRegressor(estimators=[('rf', RandomForestRegressor(random_state=42)),
                            ('xgb',
                             XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=None, max_bin=None,
                                          max_cat_threshold=None,
                                          max_cat_to_onehot=None,
                                          max_delta_step=None, max_depth=None,
                                          max_leaves=None,
                                          min_child_weight=None, missing=nan,
                                          monotone_constraints=None,
                                          multi_strategy=None, n_estimators=100,
                                          n_jobs=None, num_parallel_tree=None,
                                          random_state=42, ...)),
                            ('sgd', SGDRegressor())])

In [ ]:
# Evaluate individual models
models = {'RandomForest': rf, 'XGBoost': xgb, 'SGDRegressor': sgd}
best_model = None
best_mae = float('inf')

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    print(f'{name} MAE: {mae}')
    if mae < best_mae:
        best_mae = mae
        best_model = name
print(f'Best model based on MAE: {best_model}')

RandomForest MAE: 19.797309962156042
XGBoost MAE: 192.5530759121294
SGDRegressor MAE: 7.209043833199466e+20
Best model based on MAE: RandomForest


In [ ]:
# Make predictions using Voting Regressor
y_pred = voting_regressor.predict(X_test)

In [ ]:
# Evaluate Voting Regressor
mae = mean_absolute_error(y_test, y_pred)
print(f'Voting Regressor MAE: {mae}')

Voting Regressor MAE: 5.3464605061956226e+20
